# Assignment 2 · Classification you can defend
## Library laptop loans: which ones come back late?
**DATA3010 · put your name here:** Luna DesRoches

Read `A2_Assignment.md` for the full instructions, the marks, and the rules on AI use.

The data cell below is fixed. Do not change it: everyone's numbers should match.

Write your code under each task. Write your answers in the answer cells. Submit this notebook with its outputs.

In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
COST_LATE = 25  # dollars of disruption when a late laptop is not flagged
COST_EMAIL = 1  # dollars of staff time per reminder email

print("Setup complete.")

## The data
The library lends laptops. Some come back late, which blocks the next student and costs the desk about **\$25** of disruption. A reminder email to a borrower costs about **\$1** of staff time.

| Column | Meaning |
|---|---|
| `loan_id` | Loan number |
| `past_late_returns` | How many times this borrower returned late before |
| `loan_hours` | Hours the laptop was booked for |
| `program_year` | Year of study, 1 to 4 |
| `distance_to_campus_km` | Borrower's distance from campus |
| `renewed_online` | Whether the borrower renewed the loan online (1 = yes) |
| `campus` | Campus name |
| `fine_issued` | Whether a fine was issued for this loan (1 = yes) |
| `returned_late` | **The answer:** 1 = returned late, 0 = on time |

In [ ]:
# Fixed data cell. Do not change it.
rng = np.random.default_rng(2026)
n_loans = 900
returned_late = rng.binomial(1, 0.15, n_loans)


def loan_column(mean_late, sd_late, mean_ontime, sd_ontime, low, high, decimals=0):
    values = np.where(
        returned_late == 1,
        rng.normal(mean_late, sd_late, n_loans),
        rng.normal(mean_ontime, sd_ontime, n_loans),
    )
    return np.round(np.clip(values, low, high), decimals)


loans = pd.DataFrame(
    {
        "loan_id": [f"L{i:04d}" for i in range(1, n_loans + 1)],
        "past_late_returns": loan_column(2.1, 1.5, 0.4, 0.8, 0, 12).astype(int),
        "loan_hours": loan_column(26, 10, 8, 6, 2, 72).astype(int),
        "program_year": rng.integers(1, 5, n_loans),
        "distance_to_campus_km": loan_column(9, 6, 4, 3, 0.2, 40, 1),
        "renewed_online": rng.binomial(1, 0.4, n_loans),
        "campus": "Main",
        "fine_issued": np.where(returned_late == 1, rng.binomial(1, 0.9, n_loans), 0),
        "returned_late": returned_late,
    }
)

print("Rows, columns:", loans.shape)
display(loans.head())

## Task 1 · Inspect and choose the features (15 marks)

Look at every column and decide whether it can be a model input.

**Write code that shows:** the class balance, the baseline accuracy, and the number of different values in each column.

**Then create** `feature_cols`: the columns you will use, and `X`, `y`.

In [ ]:
# a) class balance and baseline accuracy
print("Class balance (counts):")
print(loans["returned_late"].value_counts())
print("\nClass balance (proportions):")
print(loans["returned_late"].value_counts(normalize=True))

baseline_acc = (loans["returned_late"] == 0).mean()
print(f"\nBaseline accuracy (majority class): {baseline_acc:.4f}")

# b) loans.nunique() to see which columns carry information
print("\nUnique values per column:")
print(loans.nunique())

# c) feature_cols = [...]   then  X = loans[feature_cols]   and  y = loans["returned_late"]
feature_cols = [
    "past_late_returns",
    "loan_hours",
    "program_year",
    "distance_to_campus_km",
    "renewed_online",
]
X = loans[feature_cols]
y = loans["returned_late"]


**Your answer:**


`loan_id` is a unique identifier that will cause overfitting.
`campus` is a constant value and would not affect the analysis.
`fine_issued` would introduce target leakage; fines are only issued if the laptop is returned late

I chose to keep `past_late_returns`, `loan_hours`, `program_year`, `distance_to_campus_km`, and `renewed_online`
- 755 on-time returns vs. 145 late returns. 
- Baseline accuracy: 83.89% when predicting false


## Task 2 · Split the data (10 marks)

Split into training and test sets with `test_size=0.25`, `random_state=RANDOM_STATE`, and `stratify=y`. Report the size of each part and the baseline accuracy on the test rows.

In [ ]:
# Split the data into train and test sets (75/25), stratified by y
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print(f"Training set size: {len(X_train)} rows ({len(X_train) / len(loans):.1%})")
print(f"Test set size: {len(X_test)} rows ({len(X_test) / len(loans):.1%})")
print("\nTest class distribution:")
print(y_test.value_counts())

test_baseline_acc = (y_test == 0).mean()
print(f"\nTest baseline accuracy: {test_baseline_acc:.4f}")


**Your answer:**

Training set: 675 rows
Test set (225 rows): 189 on-time; 36 late returns.

Baseline accuracy on test rows: 
84.00%, achieved by predicting the majority class (0, on-time) for all test cases.


## Task 3 · Build three classifiers (20 marks)

Fit all three on the training rows:
1. KNN inside a pipeline with `StandardScaler`
2. A decision tree
3. Logistic regression inside a pipeline with `StandardScaler`

Report accuracy, precision, recall and F1 for each on the test rows, in one table.

In [ ]:
# Fit three classifiers on the training rows
knn = make_pipeline(StandardScaler(), KNeighborsClassifier())
tree = DecisionTreeClassifier(random_state=RANDOM_STATE)
lr = make_pipeline(StandardScaler(), LogisticRegression(random_state=RANDOM_STATE))

models = {
    "KNN": knn,
    "Decision Tree": tree,
    "Logistic Regression": lr,
}

results_task3 = []
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    results_task3.append(
        {
            "Model": name,
            "Accuracy": accuracy_score(y_test, preds),
            "Precision": precision_score(y_test, preds),
            "Recall": recall_score(y_test, preds),
            "F1": f1_score(y_test, preds),
        }
    )

results_task3_df = pd.DataFrame(results_task3)
display(results_task3_df)


**Your answer:**
```
Model	Accuracy	Precision	Recall	F1
0	KNN	0.946667	1.000000	0.666667	0.800000
1	Decision Tree	0.933333	0.756098	0.861111	0.805195
2	Logistic Regression	0.955556	0.906250	0.805556	0.852941
```

## Task 4 · Tune one setting with cross-validation (15 marks)

Pick **one** setting to tune: `n_neighbors` for KNN or `max_depth` for the tree.

Try at least five values. For each one report the training accuracy, the cross-validated score on the **training rows** (`cv=5`), and the test score. Choose a value using the cross-validation column and say why.

In [ ]:
# Tune max_depth for Decision Tree using 5-fold cross-validation
depth_values = [
    1,
    2,
    3,
    4,
    5,
]
tuning_results = []

for d in depth_values:
    dt = DecisionTreeClassifier(max_depth=d, random_state=RANDOM_STATE)
    dt.fit(X_train, y_train)
    train_acc = dt.score(X_train, y_train)
    cv_acc = cross_val_score(dt, X_train, y_train, cv=5).mean()
    test_acc = dt.score(X_test, y_test)
    tuning_results.append(
        {
            "max_depth": d,
            "Training Accuracy": train_acc,
            "CV Score (cv=5)": cv_acc,
            "Test Score": test_acc,
        }
    )

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df)


**Your answer:**

- I tuned `max_depth` between 5 values with a decision tree.
- `max_depth = 3` was my choice for best_depth
- A max depth of 3 has the highest cv=5 score on the training rows of 96%. While the accuracy may have gotten higher, the other depths either over or underfit the data. 

## Task 5 · Choose a threshold using the costs (15 marks)

Take the probabilities from logistic regression (`predict_proba`).

For at least five thresholds, report how many loans you flag, how many late returns you catch, how many you miss, and the total cost:

`cost = missed * COST_LATE + flagged * COST_EMAIL`

Choose a threshold and justify it with the cost column.

In [ ]:
# Evaluate costs across multiple thresholds for Logistic Regression
lr_probs = lr.predict_proba(X_test)[:, 1]
threshold_values = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.50]

threshold_results = []
for t in threshold_values:
    flagged = int((lr_probs >= t).sum())
    caught = int(((lr_probs >= t) & (y_test == 1)).sum())
    missed = int(((lr_probs < t) & (y_test == 1)).sum())
    cost = missed * COST_LATE + flagged * COST_EMAIL
    threshold_results.append(
        {
            "Threshold": t,
            "Flagged": flagged,
            "Caught": caught,
            "Missed": missed,
            "Total Cost ($)": cost,
        }
    )

threshold_df = pd.DataFrame(threshold_results)
display(threshold_df)


**Your answer:**

- I chose a threshold of 0.10 for this model. 
- This threshold has the lowest cost and the least false negatives. 

## Task 6 · Compare the models and look at the errors (15 marks)

Put all three models in one table: accuracy, precision, recall, F1, missed late returns, emails sent, total cost. Use your chosen threshold for logistic regression.

Then print three loans your chosen model gets wrong and describe what they have in common.

In [ ]:
# Compare all three models in one table
knn_preds = knn.predict(X_test)
tree_preds = tree.predict(X_test)
chosen_threshold = 0.10
lr_preds_chosen = (lr_probs >= chosen_threshold).astype(int)


def summarize_model(name, y_pred):
    flagged = int((y_pred == 1).sum())
    missed = int(((y_pred == 0) & (y_test == 1)).sum())
    cost = missed * COST_LATE + flagged * COST_EMAIL
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "Missed late returns": missed,
        "Emails sent": flagged,
        "Total cost ($)": cost,
    }


comparison_df = pd.DataFrame(
    [
        summarize_model("KNN (default)", knn_preds),
        summarize_model("Decision Tree (default)", tree_preds),
        summarize_model(
            f"Logistic Regression (threshold = {chosen_threshold})", lr_preds_chosen
        ),
    ]
)
display(comparison_df)

# Print 3 loans the chosen model gets wrong
error_mask = y_test.values != lr_preds_chosen
error_loans = loans.loc[
    X_test.index[error_mask], ["loan_id"] + feature_cols + ["returned_late"]
].copy()
error_loans["predicted"] = lr_preds_chosen[error_mask]
error_loans["predicted_prob"] = lr_probs[error_mask]
display(error_loans.head(3))


**Your answer:**

**Misclassified loans:**
- All 3 misclassified loans are false positives. 
- Their risk signals were all above 0.10 due to past late returns. 
- Because of the price of emails ($25), the model marks the loans as false to avoid spending money on a loss which is only $1. 


## Task 7 · Choose a model and answer the AI (10 marks)

**a) Your choice.** In about 200 words, name the model and the threshold you would hand to the library, using at least three numbers from your tables. Say what would make you change your mind.

**b) The AI analysis.** A classmate asks an AI assistant for help and gets this back:

> "I built the model on all the data and it reached 97.8% accuracy, which is excellent. I included every available column so the model has as much information as possible, including `fine_issued`, which turned out to be very predictive. I used the default threshold of 0.5 because that is standard, and I chose this model because it had the highest accuracy on the test set. The library can use it as it is."

Find **at least three** things wrong with that paragraph. For each one: what is wrong, why it matters, and what you would do instead.

**Your answer:**

**a)**
I would hand the library the Logistic Regression model with a decision threshold of 0.10. The model achieved the lowest cost out of all the models. Only costing $79 on the test set of 255 loans. Compared to the $166 for Decision Tree, $324 for KNN, and $207 for Logistic Regression at a threshold of 0.50. Because late returns cause a $25 disruption cost, and the reminder emails only cost $1, a threshold of 0.10 prioritizes recall instead of accuracy. It successfully caught 35 out of 36 late returns; giving the model a 97% recall. 

I would change my mind if the:
- Operational costs increased significantly: I would recommend a higher threshold of 0.25 or 0.5 to prevent false positives. 
- If non-linear features proved to be more significant than predicted during training. 

**b)**
1. When the classmate provided all features, including `fine_issued` it caused a leakage issue in the model. The `fine_issued` feature is only filled once the decision has been make. 
2. Training the model on all the data causes the model to already see the data before it is even asked to predict the result. This causes the model to overestimate and overfit the data. 
3. The model should not only be chosen based on the highest accuracy. You should try to evaluate a models preformance using Recall, Precision, F1-score, and total cost. 

## Before you submit
- Restart the kernel and run all cells, top to bottom, with no errors.
- Every task has code, output, and a written answer.
- If you used an AI tool, say where and for what, in one or two lines below.

**AI use statement:**




I used AI to assist me in debugging, reviewing my code, and understanding the concepts in this assignment. 